# 10. Chatbot RAG Agent 구현

**OpenAI Agents SDK**로 채팅 에이전트를 단계별로 확장합니다.
도구 없는 단순 챗봇에서 시작해 날씨/웹검색/RAG 도구를 하나씩 붙여가며 완성합니다.

```
1. 기본 대화         Agent + Runner.run()
      ↓
2. 스트리밍          Runner.run_streamed() + 토큰 단위 출력
      ↓
3. 도구 추가         + get_weather + WebSearchTool
      ↓
4. RAG 추가          + FileSearchTool (Vector Store)
      ↓
5. 웹 UI            → Streamlit app.py
```

| 단계 | 내용 | 핵심 API / 도구 |
|------|------|-----------------|
| 1. 비동기 방식 호출 | 도구 없이 Agent + Runner로 대화 구현 (메시지 유지) | `await Runner.run()` |
| 2. Streaming 방식 호출 | 동일 에이전트를 토큰 단위 실시간 출력으로 실행 | `Runner.run_streamed()`, `ResponseTextDeltaEvent` |
| 3. 도구를 추가한 Chatbot | 날씨 조회 + 웹 검색 도구를 에이전트에 연결 | `@function_tool` get_weather, `WebSearchTool()` |
| 4. RAG 도구 추가 | OpenAI Vector Store에 FAQ/KB 업로드 후 검색 도구 연결 | `FileSearchTool(vector_store_ids=[...])` |
| 5. 웹 UI 연결 | 동일 에이전트를 Streamlit 웹 앱으로 구현한 예제 안내 | `10_chat-app-rag/app.py` |

### 비동기 방식 호출
도구 없이 Agent + Runner로 대화 구현 

In [ ]:
    # 사용자 입력 받기
    # 'exit' 입력 시 종료
    # 현재 사용자의 발화를 대화 기록에 추가
    # 모델의 응답을 대화 기록에 추가 (assistant 역할)
    # 모델의 최종 응답 출력

### Streaming 방식 호출

In [ ]:
#  모델의 답변이 완성될 때까지 기다리지 않고, 한 토큰씩 실시간으로 출력되는 "스트리밍" 방식
    # 현재 사용자 발화를 messages 리스트에 추가
    # 에이전트의 답변 출력 시작
    # run_streamed: 모델이 응답을 실시간으로 보낼 수 있도록 함
    # 결과(result)는 이벤트 스트림(event stream)을 포함하며,
    # 이를 async for 루프를 통해 한 토큰씩 처리합니다.
    # ResponseTextDeltaEvent: 모델이 생성 중인 텍스트 일부를 전달하는 이벤트 타입
    # event.type == "raw_response_event" 일 때,
    # event.data.delta 에는 모델이 방금 생성한 텍스트 조각이 들어 있음
    # 모델의 전체 응답을 대화 기록에 추가

### 도구를 추가한 Chatbot Agent

WebSearchTool - 모델이 응답을 생성하기 전에 웹에서 최신 정보를 검색할 수 있도록 허용합니다.  
https://developers.openai.com/api/docs/guides/tools-web-search

In [ ]:
def get_weather(위도: float, 경도: float) -> str:

### RAG(검색 증강 생성) 도구 추가

**RAG**는 지식 베이스(FAQ, 문서)를 검색해 사용자 질문에 관련된 내용을 맥락으로 제공하는 방식입니다.
여기서는 **OpenAI 관리형 Vector Store**에 FAQ와 지식 베이스 파일을 업로드하고,
에이전트의 **`FileSearchTool`** 로 연결합니다. 별도 임베딩 코드 없이 OpenAI가 파싱/청킹/임베딩/검색을 모두 처리합니다.

- **셀 1 (초기 설정)**: Vector Store를 생성하고 `faqs.json`, `knowledgeBase.json`을 업로드 (최초 1회)
- **셀 2 (에이전트)**: `FileSearchTool(vector_store_ids=[vs_id])`를 도구로 추가해 에이전트 실행

In [ ]:
# ── Vector Store 초기 설정 (최초 1회 실행) ──
# faqs.json / knowledgeBase.json 을 OpenAI Vector Store에 업로드합니다.
# 이미 같은 이름의 Vector Store가 있으면 재사용하고 재업로드하지 않습니다.
# 같은 이름의 Vector Store가 이미 있으면 재사용

```
faqs.json
[
  {"question": "비밀번호를 어떻게 재설정하나요?", "answer": "비밀번호를 재설정하려면 계정 설정 페이지에서 '비밀번호 찾기'를 클릭하세요."},
  {"question": "환불 정책이 어떻게 되나요?", "answer": "구매일로부터 30일 이내 환불이 가능합니다. 주문 번호와 함께 고객 지원팀에 문의하세요."},
  {"question": "전화 상담을 제공하나요?", "answer": "네. 월요일부터 금요일 오전 10시부터 오후 6시(IST)까지 전화 상담이 가능합니다."},
  {"question": "이메일 주소는 어떻게 변경하나요?", "answer": "프로필 설정의 이메일 환경설정에서 이메일을 변경할 수 있습니다."},
  {"question": "제 데이터는 안전한가요?", "answer": "네. 모든 사용자 데이터는 업계 표준 보안 방식으로 암호화되어 보호됩니다."}
]
```

```
kowledgeBase.json
[
  {"question": "환불 정책이 어떻게 되나요?", "answer": "구매일로부터 30일 이내 모든 상품을 전액 환불받을 수 있습니다."},
  {"question": "주문은 어디서 조회하나요?", "answer": "웹사이트의 주문 조회 페이지에서 주문 번호로 조회할 수 있습니다."}
]

```

In [ ]:
# 대화 히스토리 저장 리스트 (멀티턴 대화를 위해 메시지를 누적)
# 여행 및 FAQ 처리 에이전트 정의
# 사용자가 'exit' 입력 시 종료되는 대화 루프
    # 사용자 메시지를 대화 히스토리에 추가
    # 스트리밍 방식으로 에이전트 실행 (전체 대화 히스토리 전달)
    # 스트리밍 이벤트를 실시간으로 처리
        # 텍스트 델타 이벤트만 필터링하여 출력
    # 완성된 에이전트 응답을 대화 히스토리에 추가

---------------
### 웹 UI 구현으로 연결

동일한 에이전트 구조를 **Streamlit 웹 앱**으로 구현한 예제가 **`01_OpenAI_API_Basic/10_chat-app-rag/app.py`** 입니다.
해당 앱에서는 채팅 화면, OpenAI로 답변 생성, 대화 기록 유지까지 포함합니다.

> 참고: `app.py`는 커스텀 임베딩(`rag.py`) 방식을 사용합니다. 위 노트북의 Vector Store 방식과 비교해 볼 수 있습니다.

실행: `10_chat-app-rag` 폴더에서 `streamlit run app.py`